<a href="https://colab.research.google.com/github/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots/blob/feature%2Fdata-acquisition/Project_Group_%2343_Notebook_Submission_v01%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS1090B — Hallucination in Legal RAG Chatbots — End-to-End Pipeline

Runs the data pipeline described in the README on Google Colab Pro A100.
Stages: bootstrap → env check → CourtListener ingest → data audit → dataset probe → LePaRD ingest → LePaRD↔CL compat audit.

In [ ]:
# Cell 2: Bootstrap repo + uv venv (pipeline cells will subprocess via .venv/bin/python)
import os, subprocess, pathlib

REPO_URL = "https://github.com/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots.git"
REPO_DIR = "/content/cs1090b_HallucinationLegalRAGChatbots"

def sh(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True)
    if check and r.returncode != 0:
        raise SystemExit(f"command failed: {cmd}")

if not pathlib.Path(REPO_DIR).exists():
    sh(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

if subprocess.run("command -v uv", shell=True).returncode != 0:
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
    os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

sh("uv python install 3.11.9")
sh("uv python pin 3.11.9")
if not pathlib.Path(".venv").exists():
    sh("uv sync")

pathlib.Path(".env").write_text(
    "export PYTHONHASHSEED=0\n"
    "export CUBLAS_WORKSPACE_CONFIG=:4096:8\n"
    "export TOKENIZERS_PARALLELISM=false\n"
    "export RANDOM_SEED=0\n"
    "export TARGET_GPU_COUNT=1\n"
    "export TARGET_VRAM_GB_MIN=20.0\n"
)

# Verify .venv has correct pinned versions
r = subprocess.run(
    [".venv/bin/python", "-c",
     "import sys, numpy, torch, transformers; "
     "print(f'python {sys.version.split()[0]}'); "
     "print(f'numpy {numpy.__version__}'); "
     "print(f'torch {torch.__version__}'); "
     "print(f'transformers {transformers.__version__}')"],
    capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr); raise SystemExit("venv verification failed")

$ git clone https://github.com/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots.git /content/cs1090b_HallucinationLegalRAGChatbots
cwd: /content/cs1090b_HallucinationLegalRAGChatbots
$ uv python install 3.11.9
$ uv python pin 3.11.9
$ uv sync
python 3.11.9
numpy 1.26.4
torch 2.0.1+cu117
transformers 4.41.2



In [ ]:
# Cell 3: Environment verification — runs notebooks/cs1090b_HallucinationLegalRAGChatbots.md Cell 1
import subprocess, os, sys, re, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

nb_src = pathlib.Path("notebooks/cs1090b_HallucinationLegalRAGChatbots.md").read_text()
blocks = re.findall(r"```\{code-cell\}[^\n]*\n(.*?)```", nb_src, flags=re.DOTALL)
assert blocks, "no code cells found in notebook md"
cell1_code = blocks[0].replace(
    "os.chdir(os.path.dirname(os.getcwd()))", "# chdir not needed — already at repo root"
)

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(
    [".venv/bin/python", "-u", "-c", cell1_code],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True, env=env,
)
for line in proc.stdout:
    sys.stdout.write(line); sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"Cell 3 failed with exit code {proc.returncode}")

  [repro] Reproducibility configured:
    PYTHONHASHSEED=0
    CUBLAS_WORKSPACE_CONFIG=:4096:8
    TOKENIZERS_PARALLELISM=false
    deterministic_algorithms=True
    cudnn_benchmark=False
    cudnn_deterministic=True
    random_seed=0
    torch.cuda.manual_seed_all(0) → 1 GPU(s)
    TDD RED→GREEN: Environment Contract
  ✓ PASS: Every required dependency must be importable and meet version constraints
  ✓ PASS: CUDA GPU must be detected for training
  ✓ PASS: GPU must have at least 10GB VRAM for transformer fine-tuning
  ✓ PASS: PyTorch must be compiled with CUDA support
  ✓ PASS: Cross-dependency version constraints must be satisfied
  
    Preflight Checks — Failure Isolation Gate
  ✓ PASS: GPU count 1 (allocated)
  ✓ PASS: GPU[0] NVIDIA A100-SXM4-80GB | cap (8, 0) | 85.1GB
  ✓ PASS: torch CUDA runtime 11.7
  ✓ PASS: Disk 201.9GB free
  ✓ PASS: src/repro.py importable
  ✓ PASS: repro_cfg['PYTHONHASHSEED'] = '0'
  ✓ PASS: repro_cfg['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
  ✓ PASS: repro

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Check free space on Drive
import shutil
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive free: {free/1e9:.1f} GB / total: {total/1e9:.1f} GB")

# Create target dir:
import pathlib
dst = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_bulk")
dst.mkdir(parents=True, exist_ok=True)
print(dst, "ready")

Mounted at /content/drive
Drive free: 133.3 GB / total: 259.7 GB
/content/drive/MyDrive/cs1090b_cl_bulk ready


In [ ]:
# Replace local dir with symlink to Drive
import pathlib
src = pathlib.Path("/content/cs1090b_HallucinationLegalRAGChatbots/data/raw/cl_bulk")
dst = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_bulk")

if src.exists() and not src.is_symlink():
    src.rmdir()
src.symlink_to(dst, target_is_directory=True)

print(f"{src} -> {src.resolve()}")
for p in sorted(dst.glob("*.csv.bz2")):
    print(f"  {p.name}  {p.stat().st_size/1e9:.2f} GB")

/content/cs1090b_HallucinationLegalRAGChatbots/data/raw/cl_bulk -> /content/drive/MyDrive/cs1090b_cl_bulk
  courts-2026-03-31.csv.bz2  0.00 GB
  dockets-2026-03-31.csv.bz2  4.98 GB
  opinion-clusters-2026-03-31.csv.bz2  2.45 GB
  opinions-2026-03-31.csv.bz2  54.19 GB


In [ ]:
# Cell 4: CourtListener bulk CSV download — Drive-persistent, skip if present
import os, sys, subprocess, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_BULK = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_bulk")
DRIVE_BULK.mkdir(parents=True, exist_ok=True)

local_bulk = pathlib.Path("data/raw/cl_bulk")
local_bulk.parent.mkdir(parents=True, exist_ok=True)
if local_bulk.exists() and not local_bulk.is_symlink():
    if local_bulk.is_dir() and not any(local_bulk.iterdir()):
        local_bulk.rmdir()
    else:
        raise SystemExit(f"{local_bulk} exists and is not empty/symlink")
if not local_bulk.exists():
    local_bulk.symlink_to(DRIVE_BULK, target_is_directory=True)
print(f"bulk_dir -> {local_bulk.resolve()}")

code = r'''
import logging
from src.config import PipelineConfig
from src.s3_discovery import discover_latest_bulk_files
from src.bulk_download import download_bulk_csvs

logging.basicConfig(level=logging.INFO, format="  %(message)s")
log = logging.getLogger("bulk")

cfg = PipelineConfig()
cfg.bulk_dir.mkdir(parents=True, exist_ok=True)

need = {"courts-", "dockets-", "opinion-clusters-", "opinions-"}
have = {p.name for p in cfg.bulk_dir.glob("*.csv.bz2")}
matched = {lbl for lbl in need if any(n.startswith(lbl) for n in have)}

if matched == need:
    log.info("All 4 bulk CSVs already present in %s - skipping" % cfg.bulk_dir)
    for p in sorted(cfg.bulk_dir.glob("*.csv.bz2")):
        log.info("  %s  %.2f GB" % (p.name, p.stat().st_size / 1e9))
else:
    log.info("Missing: %s" % sorted(need - matched))
    latest = discover_latest_bulk_files(config=cfg)
    for label, info in latest.items():
        log.info("  %-10s %s (%.1f MB)" % (label, info["key"], info["size_mb"]))
    paths = download_bulk_csvs(latest, config=cfg, logger=log)
    for label, p in paths.items():
        log.info("  %-10s -> %s" % (label, p))
'''

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(
    [".venv/bin/python", "-u", "-c", code],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True, env=env,
)
for line in proc.stdout:
    sys.stdout.write(line); sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"Cell 4 failed with exit code {proc.returncode}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bulk_dir -> /content/drive/MyDrive/cs1090b_cl_bulk
  All 4 bulk CSVs already present in data/raw/cl_bulk - skipping
    courts-2026-03-31.csv.bz2  0.00 GB
    dockets-2026-03-31.csv.bz2  4.98 GB
    opinion-clusters-2026-03-31.csv.bz2  2.45 GB
    opinions-2026-03-31.csv.bz2  54.19 GB


In [ ]:
# Cell 5: Filter chain + extraction + manifest (Drive-persistent shards)
import os, sys, subprocess, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Symlink shard_dir to Drive so extraction output persists
DRIVE_SHARDS = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_federal_appellate_bulk")
DRIVE_SHARDS.mkdir(parents=True, exist_ok=True)

local_shards = pathlib.Path("data/raw/cl_federal_appellate_bulk")
local_shards.parent.mkdir(parents=True, exist_ok=True)
if local_shards.exists() and not local_shards.is_symlink():
    if local_shards.is_dir() and not any(local_shards.iterdir()):
        local_shards.rmdir()
    else:
        raise SystemExit(f"{local_shards} exists and is not empty/symlink")
if not local_shards.exists():
    local_shards.symlink_to(DRIVE_SHARDS, target_is_directory=True)
print(f"shard_dir -> {local_shards.resolve()}")

code = r'''
import logging
from src.config import PipelineConfig
from src.pipeline import run_pipeline, validate_pipeline
from src.exceptions import PipelineError
from src.timer import cell_timer

logger = logging.getLogger("cell5")
logger.setLevel(logging.INFO)
h = logging.StreamHandler()
h.setFormatter(logging.Formatter("  %(message)s"))
logger.addHandler(h)

with cell_timer("Cell 5 - Pipeline (filter + extract + manifest)", logger=logger):
    cfg = PipelineConfig()
    logger.info("=" * 60)
    logger.info("  run_pipeline")
    logger.info("=" * 60)
    manifest = run_pipeline(config=cfg, logger=logger)

    logger.info("\n" + "=" * 60)
    logger.info("  validate_pipeline")
    logger.info("=" * 60)
    try:
        validate_pipeline(config=cfg, manifest_data=manifest, logger=logger, shard_strategy="sample")
        logger.info("OK contract tests passed")
    except PipelineError as e:
        logger.error("FAIL %s" % e)
        raise

    logger.info("\n" + "=" * 60)
    logger.info("  Summary")
    logger.info("=" * 60)
    logger.info("  shards:   %d" % manifest["num_shards"])
    logger.info("  cases:    %s" % format(manifest["num_cases"], ","))
    logger.info("  scanned:  %s" % format(manifest.get("scanned", 0), ","))
    logger.info("  circuits: %d" % len(manifest.get("court_distribution", {})))
    tls = manifest.get("text_length_stats", {})
    if tls:
        logger.info("  text len: mean=%s median=%s p95=%s" % (
            format(tls.get("mean", 0), ","),
            format(tls.get("median", 0), ","),
            format(tls.get("p95", 0), ","),
        ))
'''

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(
    [".venv/bin/python", "-u", "-c", code],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True, env=env,
)
for line in proc.stdout:
    sys.stdout.write(line); sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"Cell 5 failed with exit code {proc.returncode}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
shard_dir -> /content/drive/MyDrive/cs1090b_cl_federal_appellate_bulk
    run_pipeline
  STEP 1: Discovering bulk files on S3...
    courts       bulk-data/courts-2026-03-31.csv.bz2
    dockets      bulk-data/dockets-2026-03-31.csv.bz2
    clusters     bulk-data/opinion-clusters-2026-03-31.csv.bz2
    opinions     bulk-data/opinions-2026-03-31.csv.bz2
  
STEP 2: Downloading bulk CSVs...
    ✓ courts-2026-03-31.csv.bz2 exists, skipping
    ✓ dockets-2026-03-31.csv.bz2 exists, skipping
    ✓ opinion-clusters-2026-03-31.csv.bz2 exists, skipping
    ✓ opinions-2026-03-31.csv.bz2 exists, skipping
  
STEP 3: Building filter chain...
  Loading courts...
    Federal appellate courts: 13
      ca1        Court of Appeals for the First Circuit
      ca10       Court of Appeals for the Tenth Circuit
      ca11       Court of Appeals for the Eleventh Circuit
      ca2   